# Empirical Asset Pricing via Machine Learning
**ArXivist reproduction notebook** — Gu, Kelly, Xiu (RFS 2020)

> Uses synthetic data. CRSP/WRDS required for exact replication (see `data/README_data.md`).

**Key facts**: 920 predictors, 13 ML models, 60-year panel, annual refitting, 30-year OOS test.
Best model: NN3 (3 hidden layers [32,16,8]), Sharpe 1.35 on long-short decile portfolio.


In [ ]:
import sys, os, subprocess
repo_root = os.path.abspath('..')
sys.path.insert(0, os.path.join(repo_root, 'src'))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', repo_root, '-q'], capture_output=True, text=True)
if r.returncode != 0: print('Install error:', r.stderr)
else: print('Package installed')
import numpy as np
print(f'NumPy {np.__version__}')
try:
    import torch; print(f'PyTorch {torch.__version__}')
except ImportError:
    print('PyTorch not found - NN demos will be skipped')


## Feature Construction (Section 2.1)
920-dim predictor: $z_{i,t} = x_t \otimes c_{i,t}$ (Kronecker product) + 74 industry dummies.
Characteristics are cross-sectionally ranked to $[-1,1]$ each month.


In [ ]:
from asset_pricing_ml.data.features import FeatureBuilder
import numpy as np
np.random.seed(42)
fb = FeatureBuilder()
c_raw = np.random.randn(100, 94)
x_macro = np.random.randn(8) * 0.3
ind = np.eye(74)[np.random.randint(0, 74, 100)]
z = fb.build_full_feature_vector(c_raw, x_macro, ind)
print(f'Feature vector: {z.shape}  (expected [100, 920])')
c_ranked = fb.cross_sectional_rank(c_raw)
print(f'Ranked chars range: [{c_ranked.min():.3f}, {c_ranked.max():.3f}]  (expected approx -1..1)')


## NN3 Architecture (Section 1.7)
Best performer: input [920] → 32 → 16 → 8 → output [1], ReLU + batch norm at each layer.
Ensemble over 10 random seeds (count assumed — not stated in paper).


In [ ]:
try:
    import torch
    from asset_pricing_ml.models.neural_net import FeedForwardNN
    for name, layers in [('NN1',[32]),('NN2',[32,16]),('NN3',[32,16,8]),('NN4',[32,16,8,4]),('NN5',[32,16,8,4,2])]:
        nn = FeedForwardNN(input_dim=920, hidden_layers=layers)
        n = sum(p.numel() for p in nn.parameters())
        print(f'{name}: hidden={layers}  params={n:,}')
    nn3 = FeedForwardNN(input_dim=920, hidden_layers=[32,16,8])
    z_t = torch.randn(64, 920)
    print(f'\nNN3 forward: {z_t.shape} -> {nn3(z_t).shape}')
except ImportError:
    print('PyTorch unavailable')


## Linear Methods (Sections 1.2-1.4)
OLS fails with 920 predictors (R²=-3.46%). Regularization rescues it:
elastic net (+0.11%), PCR (+0.26%), PLS (+0.27%). Still beaten by trees and NNs.


In [ ]:
from asset_pricing_ml.models.linear import OLSModel, ElasticNetModel, PCRModel, PLSModel, _oos_r2
np.random.seed(42)
N, P = 300, 920
Z_tr = np.random.randn(N, P).astype(np.float32)
R_tr = (Z_tr[:,0]*0.05 + Z_tr[:,1]*0.03 + np.random.randn(N)*0.1).astype(np.float32)
Z_vl = np.random.randn(100, P).astype(np.float32)
R_vl = (Z_vl[:,0]*0.05 + Z_vl[:,1]*0.03 + np.random.randn(100)*0.1).astype(np.float32)

paper = {'OLS':-3.46,'OLS-3':0.16,'ENet':0.11,'PCR':0.26,'PLS':0.27}
ols = OLSModel(); ols.fit(Z_tr, R_tr)
print(f'OLS:   {_oos_r2(R_vl,ols.predict(Z_vl))*100:+.3f}%  paper: {paper["OLS"]}%')
ols3 = OLSModel(n_predictors=3); ols3.fit(Z_tr, R_tr)
print(f'OLS-3: {_oos_r2(R_vl,ols3.predict(Z_vl))*100:+.3f}%  paper: {paper["OLS-3"]}%')
enet = ElasticNetModel()
enet.tune(Z_tr,R_tr,Z_vl,R_vl,[0.001,0.01,0.1],[0.0,0.5,1.0])
print(f'ENet:  {_oos_r2(R_vl,enet.predict(Z_vl))*100:+.3f}%  paper: {paper["ENet"]}%')
pcr = PCRModel(); pcr.tune(Z_tr,R_tr,Z_vl,R_vl,[1,2,5])
print(f'PCR:   {_oos_r2(R_vl,pcr.predict(Z_vl))*100:+.3f}%  paper: {paper["PCR"]}%')
pls = PLSModel(); pls.tune(Z_tr,R_tr,Z_vl,R_vl,[1,2,3])
print(f'PLS:   {_oos_r2(R_vl,pls.predict(Z_vl))*100:+.3f}%  paper: {paper["PLS"]}%')
print('(Synthetic data — quantitative comparison to paper is illustrative only)')


In [ ]:
# Tree models (Section 1.6) + portfolio construction
import time
from asset_pricing_ml.data.dataset import SyntheticDataGenerator
from asset_pricing_ml.models.trees import RandomForestModel, GBRTModel
from asset_pricing_ml.evaluation.metrics import ReturnMetrics
from asset_pricing_ml.evaluation.portfolios import PortfolioConstructor

gen = SyntheticDataGenerator(n_stocks=80, n_months=360, seed=42)
data = gen.generate()
Z_tr, R_tr = data.stack_train()
Z_vl, R_vl = data.stack_val()
Z_te, R_te = data.stack_test()
print(f'Train: {Z_tr.shape}  Val: {Z_vl.shape}  Test: {Z_te.shape}')

t0 = time.time()
rf = RandomForestModel(B=30)
rf.tune(Z_tr, R_tr, Z_vl, R_vl, L_grid=[3,5], m_grid=['sqrt'])
r2_rf = ReturnMetrics.oos_r2(R_te, rf.predict(Z_te))*100
print(f'RF  R2={r2_rf:+.4f}%  ({time.time()-t0:.1f}s)  paper: +0.33%')

t0 = time.time()
gbrt = GBRTModel()
gbrt.tune(Z_tr, R_tr, Z_vl, R_vl, L_grid=[1,2], nu_grid=[0.05], B_grid=[100])
r2_gbrt = ReturnMetrics.oos_r2(R_te, gbrt.predict(Z_te))*100
print(f'GBRT R2={r2_gbrt:+.4f}%  ({time.time()-t0:.1f}s)  paper: +0.34%')

# Long-short portfolio
ls = PortfolioConstructor.long_short_portfolio(
    [rf.predict(s.Z) for s in data.test],
    [s.R for s in data.test],
    [s.mkt_cap for s in data.test]
)
print(f'RF Long-Short Sharpe: {ls.sharpe_ratio:.2f}  (paper NN3: 1.35)')


## Paper's Key Results

### Table 1 — Monthly R²_oos
| Model | R²_oos | Notes |
|-------|--------|-------|
| OLS (920) | −3.46% | No regularization → overfit |
| OLS-3 | +0.16% | Lewellen (2015) benchmark |
| PCR | +0.26% | Dimension reduction |
| PLS | +0.27% | Dimension reduction |
| RF | +0.33% | Tree ensemble |
| GBRT | +0.34% | Boosted trees |
| **NN3** | **+0.40%** | **Best: [32,16,8] with ReLU** |

### Portfolio Results (Table 7/8)
- NN3 long-short decile: Sharpe **1.35** (VW), **2.45** (EW)
- S&P 500 timing (NN3): Sharpe **0.77** vs **0.51** buy-and-hold
- Max monthly loss (NN5 EW): **−9.01%** (lowest among all)

### Top Predictors (Figures 4-5): all models agree on
1. **Momentum** variants (mom1m, mom12m, chmom, indmom, maxret, mom36m)
2. **Liquidity** (turn, mvel1, dolvol, ill, zerotrade, baspread)
3. **Volatility** (retvol, idiovol, beta, betasq)

## Reproducing with Real Data
1. Get CRSP via WRDS — see `data/README_data.md`
2. `python train_all.py --config configs/config.yaml`
3. `python evaluate.py --results_dir results/`
4. Check `comparison/results_comparator.md` for expected values
